In [1]:
import os
from langgraph.graph import StateGraph, END
from ai_framework.nodes import *
# from ai_framework.nodes import completedProcess,thinking_steps,generate_answer

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# def route_after_validation(state: GraphState):
#     if state["is_valid"]:
#         return "thinking"
#     else:
#         return "end"

def build_graph():
    builder = StateGraph(GraphState)
    builder.add_node("validate", validate_question)
    builder.add_node("thinking", thinking_steps)
    builder.add_node("retrive_document", generate_answer)
    builder.add_node("websearch", websearch)
    builder.add_node("suggest_questions", suggest_questions)
    builder.set_entry_point("validate")
    builder.add_conditional_edges(
        "validate",
        route_after_validation,
        {
            "thinking": "thinking",
            "end":END
        }
    )
    builder.add_edge("thinking", "retrive_document")
    builder.add_edge("retrive_document", "websearch")
    builder.add_edge("websearch", "suggest_questions")
    builder.add_edge("suggest_questions", END)
    return builder.compile()

# builder.add_node("websearch", websearch)
    # builder.add_node("followupquestion", suggest_questions)

In [5]:
# -----------------------------
# STREAM FUNCTION
# -----------------------------
def ask_question_stream(query: str):
    graph = build_graph()
    print("\n=== STREAMING START ===\n")
    final_state = []
    inputObj={"query": query,
              "filename":"reactaa.pdf",
              "messageId":"1234",
              "user_id":"21a1ff59-f04b-450e-bf46-322617dae796",  #session['user_id'] 
              "user_name":"pranay", #session['user_name'] 
              "filename":'reactaa.pdf',
              "description":"this pdf is about react javascript framework"
              }

    for step in graph.stream(inputObj):
        print('step',step)
        for node, output in step.items():
            print("-"*10)
            print("node", node)
            print('output',output)
            actual_value = next(iter(output.values()))
            print(actual_value)
            print('')
            print("-"*10)
            final_state.append({"key":node,"output":actual_value})
    print("\n=== STREAMING END ===\n")

    try:
        with open('final_state.json', 'w', encoding='utf-8') as f:
            json.dump(final_state, f, indent=4)
        print("Successfully saved final_state to final_state.json")
    except Exception as e:
        print(f"Error saving JSON: {e}")
    return final_state


In [6]:
result=ask_question_stream("whats benifits of using react over angular js?")
result


=== STREAMING START ===

validate_question llm VALID
state {'query': 'whats benifits of using react over angular js?', 'filename': 'reactaa.pdf', 'user_id': '21a1ff59-f04b-450e-bf46-322617dae796', 'description': 'this pdf is about react javascript framework', 'user_name': 'pranay', 'messageId': '1234', 'is_valid': True, 'sanity_check': {'is_valid': True, 'messageId': '1234'}}
step {'validate': {'sanity_check': {'is_valid': True, 'messageId': '1234'}, 'is_valid': True}}
----------
node validate
output {'sanity_check': {'is_valid': True, 'messageId': '1234'}, 'is_valid': True}
{'is_valid': True, 'messageId': '1234'}

----------
step {'thinking': {'thinking': {'content': '### RAG Planning Protocol for "whats benifits of using react over angular js?"\n1. **Analyze & Query**: Identify key concepts.\n2. **Retrieve**: Perform semantic search.\n3. **Augment & Generate**: Combine context for response.', 'used_tokens': 54, 'prompt_tokens': 562, 'total_tokens': 616, 'messageid': '1234'}}}
------

[{'key': 'validate', 'output': {'is_valid': True, 'messageId': '1234'}},
 {'key': 'thinking',
  'output': {'content': '### RAG Planning Protocol for "whats benifits of using react over angular js?"\n1. **Analyze & Query**: Identify key concepts.\n2. **Retrieve**: Perform semantic search.\n3. **Augment & Generate**: Combine context for response.',
   'used_tokens': 54,
   'prompt_tokens': 562,
   'total_tokens': 616,
   'messageid': '1234'}},
 {'key': 'retrive_document',
  'output': {'content': '{\n    "answer": "React offers several benefits, including improved performance through its virtual DOM, a component-based architecture for modular development, and strong community support. Additionally, React provides features like JSX for describing UI components and context for passing data through the component tree without manual prop passing.",\n    "citations": ["reactaa.pdf, Page 2", "reactaa.pdf, Page 7"]\n}',
   'used_tokens': 82,
   'prompt_tokens': 484,
   'total_tokens': 566,
   'm